# ETL — V1DD 1196 synapses (`SynapseConnectivityLong` + `SynapseFeatureMatrix`)

Writes one logical synapse table and its wide feature payload:

- **`synapse/`** — long `SynapseConnectivityLong` rows
  (`id`, endpoints, `synapse_table_id`, `project_id`).
- **`synapsefeatures/<feature_matrix_id>/`** — wide per-synapse features.
- **`synapsefeaturematrix/`** — the `SynapseFeatureMatrix` pointer row.
- **`cellcellconnectivitylong/`** — derived cell-cell measurements identified by
  `connectome_id`; this ETL also populates optional source `synapse_table_id`
  provenance.

`dataset_id = "v1dd_1196_em"` remains reserved for the DataItem collection;
`synapse_table_id = "v1dd_1196_synapses"` identifies the synapse rows.
Large dataframe-backed tables use `write_deltalake`; the metadata pointer uses
`write_models`. `read_synapse_table` reads the long table and optionally joins
its feature payload.

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
from deltalake import write_deltalake

from connects_common_connectivity.io import read_synapse_table, write_models
from connects_common_connectivity.io.arrow_utils import attach_linkml_metadata, build_arrow_schema
from connects_common_connectivity.io.path_spec import (
    CELL_CELL_CONNECTIVITY_SUBDIR,
    SYNAPSE_FEATURES_SUBDIR,
    SYNAPSE_SUBDIR,
)
from connects_common_connectivity.models import SynapseConnectivityLong, SynapseFeatureMatrix

In [ ]:
# --- Constants -------------------------------------------------------------
PROJECT_ID = "v1dd"
DATASET_ID = "v1dd_1196_em"
SYNAPSE_TABLE_ID = "v1dd_1196_synapses"
FEATURE_MATRIX_ID = "v1dd_1196_synapse_features"

OUTPUT_ROOT = Path("../scratch/v1dd_1196_v2/")

## Load source

`syn_df` is one row per synapse (`id` unique); `syn_label_df` gives a
spine/shaft/soma `tag` for a **subset** of synapses (renamed to
`synaptictargetlabel`). Root ids are 18-digit ints kept as strings — never
cast.

In [3]:
DATA_ROOT = Path("/data/v1dd_1196")
syn_df = pd.read_feather(DATA_ROOT / "syn_df_all_to_proofread_to_all_1196.feather")
for col in ("id", "pre_pt_root_id", "post_pt_root_id"):
    syn_df[col] = syn_df[col].astype(str)

label_df = (
    pd.read_feather(DATA_ROOT / "syn_label_df_all_to_proofread_to_all_1196.feather")
    .reset_index()                                  # 'id' is the feather index
    .rename(columns={"tag": "synaptictargetlabel"})
    .astype({"id": str})
)
print("syn_df:", syn_df.shape,
      "| labels cover", f"{label_df['id'].nunique()/len(syn_df):.1%}", "of synapses")

syn_df: (8204497, 13) | labels cover 81.7% of synapses


## Write 1 — long single-synapse table → `synapse/`

Long-form `SynapseConnectivityLong` columns, built straight from the dataframe
into an Arrow table (with the model's schema + LinkML metadata), then written
with a `(project_id, synapse_table_id)`-scoped overwrite. `write_models(long_rows, ...)`
is the equivalent one-liner and is the right call at smaller scale, but here it
would mean instantiating ~8M pydantic models.

In [ ]:
long_df = pd.DataFrame({
    "id": syn_df["id"],
    "presynaptic_cell": syn_df["pre_pt_root_id"],
    "postsynaptic_cell": syn_df["post_pt_root_id"],
    "synapse_table_id": SYNAPSE_TABLE_ID,
    "project_id": PROJECT_ID,
})
long_table = attach_linkml_metadata(
    pa.Table.from_pandas(long_df, schema=build_arrow_schema(SynapseConnectivityLong),
                         preserve_index=False),
    linkml_class="SynapseConnectivityLong",
)
write_deltalake(
    str(OUTPUT_ROOT / SYNAPSE_SUBDIR),
    long_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
print("long rows written:", long_table.num_rows)

long rows written: 8204497


## Write 2 — wide per-synapse feature table → `synapsefeatures/<id>/`

Built from raw dataframes (not model instances), keyed by the synapse `id`,
with `synaptictargetlabel` LEFT-joined from the label feather. This mirrors how
`cellfeatures/` wide tables are handled — outside the model registry.

In [ ]:
# Features are every syn_df column that is not the synapse id or a
# pre/post root id (those define the connection, not a feature).
feature_cols = [c for c in syn_df.columns if c != "id" and not c.endswith("_root_id")]
wide = syn_df[["id"] + feature_cols].merge(
    label_df[["id", "synaptictargetlabel"]], on="id", how="left",
)
wide["project_id"] = PROJECT_ID
wide["synapse_table_id"] = SYNAPSE_TABLE_ID
write_deltalake(
    str(OUTPUT_ROOT / SYNAPSE_FEATURES_SUBDIR / FEATURE_MATRIX_ID),
    pa.Table.from_pandas(wide, preserve_index=False),
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND "
        f"synapse_table_id = '{SYNAPSE_TABLE_ID}'"
    ),
    partition_by=["project_id"],
)
print("feature rows written:", len(wide),
      "| labelled:", int(wide["synaptictargetlabel"].notna().sum()))

feature rows written: 8204497 | labelled: 6706286


## Write 3 — `SynapseFeatureMatrix` pointer row → `synapsefeaturematrix/`

A single metadata row locating the wide feature table and naming its synapse-id
column, written through `write_models` (`(project_id, id)`-scoped per its
`WriteSpec`).

In [ ]:
feature_matrix = SynapseFeatureMatrix(
    id=FEATURE_MATRIX_ID,
    description="Per-synapse position, size and target label for V1DD 1196.",
    synapse_table_id=SYNAPSE_TABLE_ID,
    project_id=PROJECT_ID,
    parquet_path=(
        f"file://{(OUTPUT_ROOT / SYNAPSE_FEATURES_SUBDIR / FEATURE_MATRIX_ID).resolve()}/"
    ),
    synapse_index_column="id",
)
print("pointer rows written:", write_models([feature_matrix], output_root=OUTPUT_ROOT).rows_written)

pointer rows written: 1


## Verify

Read the long table back and LEFT-join the wide features. The join preserves
every synapse; unlabeled synapses get a null `synaptictargetlabel`.

In [ ]:
full = read_synapse_table(
    PROJECT_ID,
    synapse_table_id=SYNAPSE_TABLE_ID,
    features=True,
    feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=OUTPUT_ROOT,
)

In [14]:
full.head(3)

id,presynaptic_cell,postsynaptic_cell,dataset_id,project_id,pre_pt_position_x,pre_pt_position_y,pre_pt_position_z,post_pt_position_x,post_pt_position_y,post_pt_position_z,ctr_pt_position_x,ctr_pt_position_y,ctr_pt_position_z,size,synaptictargetlabel
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,str
"""354386968""","""864691132536286810""","""864691132734919083""","""v1dd_1196_em""","""v1dd""",758200.5,802316.1,304380.0,757861.0,802558.6,304650.0,757967.7,802597.4,304380.0,240,"""shaft"""
"""378070488""","""864691132572190492""","""864691132606767301""","""v1dd_1196_em""","""v1dd""",792063.2,514342.5,183735.0,792664.6,514284.3,183915.0,792412.4,514294.0,183735.0,3056,"""shaft"""
"""499493001""","""864691132573738810""","""864691132747578447""","""v1dd_1196_em""","""v1dd""",977071.3,390075.8,191340.0,976974.3,390104.9,190935.0,976838.5,390337.7,190935.0,1346,null


In [13]:
full.select(["id", "presynaptic_cell", "postsynaptic_cell", "size", "synaptictargetlabel"]).head(3)

id,presynaptic_cell,postsynaptic_cell,size,synaptictargetlabel
str,str,str,i64,str
"""354386968""","""864691132536286810""","""864691132734919083""",240,"""shaft"""
"""378070488""","""864691132572190492""","""864691132606767301""",3056,"""shaft"""
"""499493001""","""864691132573738810""","""864691132747578447""",1346,null


---

## Cell-cell connectivity from synapses → `cellcellconnectivitylong/`

Aggregate the full single-synapse table into `CellCellConnectivityLong` pairs,
restricted to the proofread axon and proofread dendrite DataItem cohorts.
`connectome_id` independently identifies this measurement context; because this
ETL derives from one source table, it also records optional `synapse_table_id`
provenance.

In [ ]:
import polars as pl

from connects_common_connectivity.io import (
    cell_cell_connectivity_to_arrow,
    derive_cell_cell_connectivity,
)
from connects_common_connectivity.models import Modality, Unit

DATASET_PROOFREAD_AXON = "v1dd_1196_proofread_axons"
DATASET_PROOFREAD_DEND = "v1dd_1196_proofread_dendrites"
CONNECTOME_ID = "v1dd_1196_em_proofread_axon_to_dendrite"

# Cohort membership (pre = proofread axons, post = proofread dendrites) comes from
# the DataItemDataSetAssociation delta written by etl_v1dd_02_cave.ipynb.
assoc = pl.read_delta(str(OUTPUT_ROOT / "dataitem_dataset_association")).filter(
    pl.col("project_id") == PROJECT_ID
)
axon_ids = set(
    assoc.filter(pl.col("dataset_id") == DATASET_PROOFREAD_AXON)["dataitem_id"].to_list()
)
dend_ids = set(
    assoc.filter(pl.col("dataset_id") == DATASET_PROOFREAD_DEND)["dataitem_id"].to_list()
)
print(f"proofread axon cells (pre): {len(axon_ids)} | "
      f"proofread dendrite cells (post): {len(dend_ids)}")

proofread axon cells (pre): 1164 | proofread dendrite cells (post): 63986


Read the per-synapse table back from the delta lake (with the `size` feature
joined in), keep only synapses whose presynaptic cell is a proofread axon and
whose postsynaptic cell is a proofread dendrite, then derive count and size
measurements with `derive_cell_cell_connectivity`.

In [ ]:
syn = read_synapse_table(
    PROJECT_ID,
    synapse_table_id=SYNAPSE_TABLE_ID,
    features=True,
    feature_matrix_id=FEATURE_MATRIX_ID,
    output_root=OUTPUT_ROOT,
)

selected_synapses = syn.filter(
    pl.col("presynaptic_cell").is_in(axon_ids)
    & pl.col("postsynaptic_cell").is_in(dend_ids)
)
pair_count = selected_synapses.select(
    "presynaptic_cell", "postsynaptic_cell"
).unique().height
print(f"connected (axon, dendrite) pairs: {pair_count} "
      f"(from {selected_synapses.height} synapses)")

connected (axon, dendrite) pairs: 739022 (from 1431746 synapses)


Derive two `CellCellConnectivityLong` rows per pair — `SYNAPSE_COUNT` (unit
`COUNT`) and `SUM_ANATOMICAL_SIZE` (unit `ARBITRARY_UNIT`) — and write them to
the canonical table with a `(project_id, connectome_id)`-scoped overwrite,
partitioned by `project_id`, `connectome_id`, and `measurement_type`.

In [ ]:
connectivity = derive_cell_cell_connectivity(
    selected_synapses,
    project_id=PROJECT_ID,
    connectome_id=CONNECTOME_ID,
    modality=Modality.ELECTRON_MICROSCOPY,
    size_column="size",
    size_unit=Unit.ARBITRARY_UNIT,
)
conn_table = attach_linkml_metadata(
    cell_cell_connectivity_to_arrow(connectivity),
    linkml_class="CellCellConnectivityLong",
)
write_deltalake(
    str(OUTPUT_ROOT / CELL_CELL_CONNECTIVITY_SUBDIR),
    conn_table,
    mode="overwrite",
    predicate=(
        f"project_id = '{PROJECT_ID}' AND connectome_id = '{CONNECTOME_ID}'"
    ),
    partition_by=["project_id", "connectome_id", "measurement_type"],
)
print(f"CellCellConnectivityLong rows written: {conn_table.num_rows} "
      f"({pair_count} pairs x 2 measurement types)")

CellCellConnectivityLong rows written: 1478044 (739022 pairs x 2 measurement types)


### Verify

Read the connectivity table back and confirm both measurement types are present
and every row survives the round-trip.

In [ ]:
conn_v = pl.read_delta(str(OUTPUT_ROOT / CELL_CELL_CONNECTIVITY_SUBDIR)).filter(
    (pl.col("project_id") == PROJECT_ID)
    & (pl.col("connectome_id") == CONNECTOME_ID)
)
measurement_counts = conn_v.group_by("measurement_type").len()
print("shape:", conn_v.shape)
print(measurement_counts)
assert conn_v.shape[0] == pair_count * 2
assert conn_v["synapse_table_id"].unique().to_list() == [SYNAPSE_TABLE_ID]
assert conn_v["connectome_id"].unique().to_list() == [CONNECTOME_ID]
assert conn_v["measurement_type"].n_unique() == 2
assert sorted(measurement_counts["len"].to_list()) == [pair_count, pair_count]
conn_v.head(3)

shape: (1478044, 9)
shape: (2, 2)
┌─────────────────────┬────────┐
│ measurement_type    ┆ len    │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ SYNAPSE_COUNT       ┆ 739022 │
│ SUM_ANATOMICAL_SIZE ┆ 739022 │
└─────────────────────┴────────┘


id,description,presynaptic_cell,postsynaptic_cell,modality,value,unit,project_id,measurement_type
str,str,str,str,str,f64,str,str,str
"""864691132683527136_86469113263…",null,"""864691132683527136""","""864691132631260245""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT""","""v1dd""","""SYNAPSE_COUNT"""
"""864691132772048788_86469113292…",null,"""864691132772048788""","""864691132920163339""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT""","""v1dd""","""SYNAPSE_COUNT"""
"""864691132641639590_86469113265…",null,"""864691132641639590""","""864691132659454181""","""ELECTRON_MICROSCOPY""",1.0,"""COUNT""","""v1dd""","""SYNAPSE_COUNT"""
